<a href="https://colab.research.google.com/github/LiJHsysu/AF2Bind-Transformer/blob/main/af2bind.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### AF2BIND: Prediction of ligand-binding sites using AlphaFold2

AF2BIND is a simple and fast notebook that runs inference on the output obtained from [AlphaFold2](https://github.com/deepmind/alphafold).

<!--<img src="https://raw.githubusercontent.com/artemg97/af2bind_prod/main/logo.png" width="300">.-->

<figure>
<img src='https://raw.githubusercontent.com/artemg97/af2bind_prod/main/logo.png'  width="300" height="150"/>
</figure>



For more details see preprint:

**AF2BIND: Predicting ligand-binding sites using the pair representation of AlphaFold2**
* Artem Gazizov, Anna Lian, Casper Alexander Goverde, Sergey Ovchinnikov, Nicholas F. Polizzi
* https://doi.org/10.1101/2023.10.15.562410


In [1]:
%%time
#@title Install AlphaFold2 (~2 mins)
#@markdown Please execute this cell by pressing the *Play* button on
#@markdown the left.
import os, time
if not os.path.isdir("params"):
  # get code
  print("installing ColabDesign")
  os.system("(mkdir params; apt-get install aria2 -qq; \
  aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2021-07-14.tar; \
  mkdir af2bind_params; \
  wget -qnc https://github.com/sokrypton/af2bind/raw/main/attempt_7_2k_lam0-03.zip; unzip attempt_7_2k_lam0-03.zip -d af2bind_params; \
  tar -xf alphafold_params_2021-07-14.tar -C params; touch params/done.txt )&")

  os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git@v1.1.1")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

  # download params
  if not os.path.isfile("params/done.txt"):
    print("downloading params")
    while not os.path.isfile("params/done.txt"):
      time.sleep(5)

import os
from colabdesign import mk_afdesign_model, clear_mem
from IPython.display import HTML
from google.colab import files
import numpy as np

from colabdesign.af.alphafold.common import residue_constants
import pandas as pd
from google.colab import data_table
data_table._DEFAULT_FORMATTERS[float] = lambda x: f"{x:.3f}"
from IPython.display import display, HTML
import jax, pickle
import jax.numpy as jnp
from scipy.special import expit as sigmoid
import plotly.express as px

import py3Dmol
import matplotlib.pyplot as plt
from scipy.special import softmax
import copy
from colabdesign.shared.protein import renum_pdb_str
from colabdesign.af.alphafold.common import protein


aa_order = {v:k for k,v in residue_constants.restype_order.items()}

def get_pdb(pdb_code=""):
  if pdb_code is None or pdb_code == "":
    upload_dict = files.upload()
    pdb_string = upload_dict[list(upload_dict.keys())[0]]
    with open("tmp.pdb","wb") as out: out.write(pdb_string)
    return "tmp.pdb"
  elif os.path.isfile(pdb_code):
    return pdb_code
  elif len(pdb_code) == 4:
    os.system(f"wget -qnc https://files.rcsb.org/view/{pdb_code}.pdb")
    return f"{pdb_code}.pdb"
  else:
    os.system(f"wget -qnc https://alphafold.ebi.ac.uk/files/AF-{pdb_code}-F1-model_v4.pdb")
    return f"AF-{pdb_code}-F1-model_v4.pdb"

def af2bind(outputs, mask_sidechains=True, seed=0):
  pair_A = outputs["representations"]["pair"][:-20,-20:]
  pair_B = outputs["representations"]["pair"][-20:,:-20].swapaxes(0,1)
  pair_A = pair_A.reshape(pair_A.shape[0],-1)
  pair_B = pair_B.reshape(pair_B.shape[0],-1)
  x = np.concatenate([pair_A,pair_B],-1)

  # get params
  if mask_sidechains:
    model_type = f"split_nosc_pair_A_split_nosc_pair_B_{seed}"
  else:
    model_type = f"split_pair_A_split_pair_B_{seed}"
  with open(f"af2bind_params/attempt_7_2k_lam0-03/{model_type}.pickle","rb") as handle:
    params_ = pickle.load(handle)
  params_ = dict(**params_["~"], **params_["linear"])
  p = jax.tree_util.tree_map(lambda x:np.asarray(x), params_)

  # get predictions
  x = (x - p["mean"]) / p["std"]
  x = (x * p["w"][:,0]) + (p["b"] / x.shape[-1])
  p_bind_aa = x.reshape(x.shape[0],2,20,-1).sum((1,3))
  p_bind = sigmoid(p_bind_aa.sum(-1))
  return {"p_bind":p_bind, "p_bind_aa":p_bind_aa}

installing ColabDesign
downloading params
CPU times: user 2.41 s, sys: 266 ms, total: 2.67 s
Wall time: 1min 5s


In [2]:
import os
import pickle
import numpy as np

In [3]:
def make_training_sample(target_pdb="6w70", target_chain="A"):

    clear_mem()

    af_model = mk_afdesign_model(
        protocol="binder",
        debug=True
    )

    af_model.prep_inputs(
        pdb_filename=f"{target_pdb}.pdb",
        chain=target_chain,
        binder_len=20,
        rm_target_seq=False
    )

    af_model.set_seq("ACDEFGHIKLMNPQRSTVWY")

    af_model.predict(verbose=False)

    outputs = af_model.aux["debug"]["outputs"]

    pair_A = outputs["representations"]["pair"][:-20, -20:]
    pair_B = outputs["representations"]["pair"][-20:, :-20].swapaxes(0,1)

    pair_A = pair_A.reshape(pair_A.shape[0], -1)
    pair_B = pair_B.reshape(pair_B.shape[0], -1)

    x = np.concatenate([pair_A, pair_B], axis=-1)

    ####################################################
    # 临时假标签（后面再换真实标签）
    ####################################################

    y = np.random.randint(0, 2, size=(x.shape[0],))

    ####################################################

    save_data = {
        "x": x.astype(np.float32),
        "y": y.astype(np.float32)
    }

    os.makedirs("data", exist_ok=True)

    save_path = f"data/{target_pdb}_{target_chain}.pkl"

    with open(save_path, "wb") as f:
        pickle.dump(save_data, f)

    print("saved:", save_path)
    print("x shape:", x.shape)
    print("y shape:", y.shape)

In [4]:
!wget https://files.rcsb.org/download/6W70.pdb

--2026-05-20 00:11:19--  https://files.rcsb.org/download/6W70.pdb
Resolving files.rcsb.org (files.rcsb.org)... 52.84.20.13, 52.84.20.16, 52.84.20.90, ...
Connecting to files.rcsb.org (files.rcsb.org)|52.84.20.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘6W70.pdb’

6W70.pdb                [ <=>                ] 524.68K  --.-KB/s    in 0.06s   

2026-05-20 00:11:20 (7.98 MB/s) - ‘6W70.pdb’ saved [537273]



In [7]:
!mv 6W70.pdb 6w70.pdb

In [8]:
make_training_sample("6w70", "A")

saved: data/6w70_A.pkl
x shape: (126, 5120)
y shape: (126,)


In [9]:
import torch
import torch.nn as nn

In [10]:
class AF2BindTransformer(nn.Module):

    def __init__(
        self,
        input_dim=5120,
        hidden_dim=256,
        num_heads=8,
        num_layers=2
    ):

        super().__init__()

        self.input_proj = nn.Linear(input_dim, hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):

        x = self.input_proj(x)

        x = self.transformer(x)

        x = self.classifier(x)

        x = x.squeeze(-1)

        return x

In [11]:
from torch.utils.data import Dataset
import pickle
import os

In [12]:
class AF2BindDataset(Dataset):

    def __init__(self, data_dir="data"):

        self.files = []

        for f in os.listdir(data_dir):

            if f.endswith(".pkl"):

                self.files.append(
                    os.path.join(data_dir, f)
                )

    def __len__(self):

        return len(self.files)

    def __getitem__(self, idx):

        with open(self.files[idx], "rb") as f:

            d = pickle.load(f)

        x = torch.tensor(d["x"], dtype=torch.float32)

        y = torch.tensor(d["y"], dtype=torch.float32)

        return x, y

In [13]:
from torch.utils.data import DataLoader

In [14]:
dataset = AF2BindDataset()

loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True
)

In [15]:
model = AF2BindTransformer()

criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [16]:
epochs = 5

model.train()

for epoch in range(epochs):

    total_loss = 0

    for x, y in loader:

        optimizer.zero_grad()

        pred = model(x)

        loss = criterion(pred, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}")
    print("loss =", total_loss)

Epoch 1
loss = 0.7218982577323914
Epoch 2
loss = 0.7505709528923035
Epoch 3
loss = 0.6839760541915894
Epoch 4
loss = 0.6880629062652588
Epoch 5
loss = 0.6942613124847412


In [17]:
torch.save(
    model.state_dict(),
    "af2bind_transformer.pt"
)

In [18]:
!mkdir -p dataset
!mkdir -p dataset/pdb
!mkdir -p dataset/pkl

In [19]:
!wget https://zhanggroup.org/BioLiP/download/BioLiP.txt.gz

--2026-05-20 00:17:01--  https://zhanggroup.org/BioLiP/download/BioLiP.txt.gz
Resolving zhanggroup.org (zhanggroup.org)... 172.67.157.77, 104.21.13.194, 2606:4700:3037::ac43:9d4d, ...
Connecting to zhanggroup.org (zhanggroup.org)|172.67.157.77|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://aideepmed.com/BioLiP/download/BioLiP.txt.gz [following]
--2026-05-20 00:17:01--  https://aideepmed.com/BioLiP/download/BioLiP.txt.gz
Resolving aideepmed.com (aideepmed.com)... 172.67.149.27, 104.21.33.198, 2606:4700:3036::ac43:951b, ...
Connecting to aideepmed.com (aideepmed.com)|172.67.149.27|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 78570940 (75M) [application/gzip]
Saving to: ‘BioLiP.txt.gz’

BioLiP.txt.gz       100%[===================>]  74.93M  21.0MB/s    in 4.5s    

2026-05-20 00:17:06 (16.6 MB/s) - ‘BioLiP.txt.gz’ saved [78570940/78570940]



In [20]:
!gunzip BioLiP.txt.gz

In [21]:
!head BioLiP.txt

101m	A	2.07	BS01	HEM	A	1	F43 R45 V68 S92 H93 H97 I99 Y103	F44 R46 V69 S93 H94 H98 I100 Y104			1.11.1.-,1.7.-.-	0004601,0005344,0005737,0015671,0016491,0016528,0019430,0019825,0020037,0046872,0070062,0098809					P02185		155 	MVLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRVKHLKTEAEMKASEDLKKHGVTVLTALGAILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISEAIIHVLHSRHPGNFGADAQGAMNKALELFRKDIAAKYKELGYQG
102m	A	1.84	BS01	HEM	A	1	F43 R45 T67 L89 S92 H93 H97 I99 Y103	F44 R46 T68 L90 S93 H94 H98 I100 Y104			1.11.1.-,1.7.-.-	0004601,0005344,0005737,0015671,0016491,0016528,0019430,0019825,0020037,0046872,0070062,0098809					P02185		155 	MVLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRFKHLKTEAEMKASEDLKKAGVTVLTALGAILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISEAIIHVLHSRHPGNFGADAQGAMNKALELFRKDIAAKYKELGYQG
103m	A	2.07	BS01	HEM	A	1	F43 R45 S92 H93 H97 I99 Y103	F44 R46 S93 H94 H98 I100 Y104			1.11.1.-,1.7.-.-	0004601,0005344,0005737,0015671,0016491,0016528,0019430,0019825,0020037,0046872,0070062,0098809					P02185		155 	MVLSEGEWQLVLHVWAKV

In [22]:
import pandas as pd

In [23]:
with open("BioLiP.txt") as f:

    lines = f.readlines()

print("total samples =", len(lines))

print(lines[0])

total samples = 989058
101m	A	2.07	BS01	HEM	A	1	F43 R45 V68 S92 H93 H97 I99 Y103	F44 R46 V69 S93 H94 H98 I100 Y104			1.11.1.-,1.7.-.-	0004601,0005344,0005737,0015671,0016491,0016528,0019430,0019825,0020037,0046872,0070062,0098809					P02185		155 	MVLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRVKHLKTEAEMKASEDLKKHGVTVLTALGAILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISEAIIHVLHSRHPGNFGADAQGAMNKALELFRKDIAAKYKELGYQG



In [25]:
!wget https://files.rcsb.org/download/6W70.pdb
!mv 6W70.pdb dataset/pdb/6w70.pdb

--2026-05-20 00:19:01--  https://files.rcsb.org/download/6W70.pdb
Resolving files.rcsb.org (files.rcsb.org)... 52.84.20.16, 52.84.20.116, 52.84.20.90, ...
Connecting to files.rcsb.org (files.rcsb.org)|52.84.20.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘6W70.pdb’

6W70.pdb                [ <=>                ] 524.68K  --.-KB/s    in 0.03s   

2026-05-20 00:19:01 (15.7 MB/s) - ‘6W70.pdb’ saved [537273]



In [26]:
from Bio.PDB import PDBParser
import numpy as np

In [27]:
def build_real_labels(pdb_file, chain_id="A", threshold=4.0):

    parser = PDBParser(QUIET=True)

    structure = parser.get_structure("protein", pdb_file)

    model = structure[0]

    chain = model[chain_id]

    residues = list(chain)

    ligand_atoms = []

    protein_atoms = []

    residue_indices = []

    ##################################################
    # 收集 ligand atoms
    ##################################################

    for c in model:

        for res in c:

            hetflag = res.id[0]

            if hetflag != " ":

                for atom in res:

                    ligand_atoms.append(atom.coord)

    ligand_atoms = np.array(ligand_atoms)

    ##################################################
    # residue label
    ##################################################

    labels = []

    for i, res in enumerate(residues):

        is_binding = 0

        for atom in res:

            dists = np.linalg.norm(
                ligand_atoms - atom.coord,
                axis=1
            )

            if np.min(dists) < threshold:

                is_binding = 1
                break

        labels.append(is_binding)

    return np.array(labels, dtype=np.float32)

In [28]:
y = build_real_labels(
    "dataset/pdb/6w70.pdb",
    chain_id="A"
)

print(y.shape)

print("binding residues =", y.sum())

(290,)
binding residues = 284.0


In [29]:
def make_real_training_sample(
    target_pdb="6w70",
    target_chain="A"
):

    ##################################################
    # AlphaFold feature
    ##################################################

    clear_mem()

    af_model = mk_afdesign_model(
        protocol="binder",
        debug=True
    )

    af_model.prep_inputs(
        pdb_filename=f"dataset/pdb/{target_pdb}.pdb",
        chain=target_chain,
        binder_len=20,
        rm_target_seq=False
    )

    af_model.set_seq("ACDEFGHIKLMNPQRSTVWY")

    af_model.predict(verbose=False)

    outputs = af_model.aux["debug"]["outputs"]

    ##################################################
    # pair representation
    ##################################################

    pair_A = outputs["representations"]["pair"][:-20, -20:]
    pair_B = outputs["representations"]["pair"][-20:, :-20].swapaxes(0,1)

    pair_A = pair_A.reshape(pair_A.shape[0], -1)
    pair_B = pair_B.reshape(pair_B.shape[0], -1)

    x = np.concatenate([pair_A, pair_B], axis=-1)

    ##################################################
    # REAL labels
    ##################################################

    y = build_real_labels(
        f"dataset/pdb/{target_pdb}.pdb",
        chain_id=target_chain
    )

    ##################################################
    # length safety
    ##################################################

    n = min(len(x), len(y))

    x = x[:n]
    y = y[:n]

    ##################################################
    # save
    ##################################################

    save_data = {
        "x": x.astype(np.float32),
        "y": y.astype(np.float32)
    }

    save_path = f"dataset/pkl/{target_pdb}_{target_chain}.pkl"

    with open(save_path, "wb") as f:

        pickle.dump(save_data, f)

    print("saved:", save_path)

    print("x shape =", x.shape)

    print("positive labels =", y.sum())

In [30]:
make_real_training_sample(
    "6w70",
    "A"
)

saved: dataset/pkl/6w70_A.pkl
x shape = (126, 5120)
positive labels = 120.0


In [36]:
targets = [
    ("6w70", "A"),
]

In [37]:
for pdb_id, chain_id in targets:

    print("processing", pdb_id)

    try:

        make_real_training_sample(
            pdb_id,
            chain_id
        )

    except Exception as e:

        print("FAILED:", e)

processing 6w70
saved: dataset/pkl/6w70_A.pkl
x shape = (126, 5120)
positive labels = 120.0


In [38]:
targets = [
    ("6w70", "A"),
    ("1a3n", "A"),
    ("1b38", "A"),
    ("1h00", "A"),
    ("2p16", "A"),
]

In [40]:
for pdb_id, chain_id in targets:

    pdb_upper = pdb_id.upper()

    cmd = f"""
    wget https://files.rcsb.org/download/{pdb_upper}.pdb
    mv {pdb_upper}.pdb dataset/pdb/{pdb_id}.pdb
    """

    os.system(cmd)

    print("downloaded", pdb_id)

downloaded 6w70
downloaded 1a3n
downloaded 1b38
downloaded 1h00
downloaded 2p16


In [41]:
for pdb_id, chain_id in targets:

    print("processing", pdb_id)

    try:

        make_real_training_sample(
            pdb_id,
            chain_id
        )

    except Exception as e:

        print("FAILED:", pdb_id)

        print(e)

processing 6w70
saved: dataset/pkl/6w70_A.pkl
x shape = (126, 5120)
positive labels = 120.0
processing 1a3n
saved: dataset/pkl/1a3n_A.pkl
x shape = (141, 5120)
positive labels = 120.0
processing 1b38
saved: dataset/pkl/1b38_A.pkl
x shape = (290, 5120)
positive labels = 227.0
processing 1h00
saved: dataset/pkl/1h00_A.pkl
x shape = (278, 5120)
positive labels = 212.0
processing 2p16
saved: dataset/pkl/2p16_A.pkl
x shape = (227, 5120)
positive labels = 171.0


In [43]:
dataset = AF2BindDataset("dataset/pkl")

In [44]:
loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True
)

In [45]:
epochs = 20

In [46]:
torch.save(
    model.state_dict(),
    "af2bind_transformer_real.pt"
)

In [47]:
all_targets = [
    ("6w70", "A"),
    ("1a3n", "A"),
    ("1b38", "A"),
    ("1h00", "A"),
    ("2p16", "A"),
]

In [48]:
train_targets = all_targets[:4]

val_targets = all_targets[4:]

print("train =", train_targets)

print("val =", val_targets)

train = [('6w70', 'A'), ('1a3n', 'A'), ('1b38', 'A'), ('1h00', 'A')]
val = [('2p16', 'A')]


In [49]:
os.makedirs("dataset/train", exist_ok=True)
os.makedirs("dataset/val", exist_ok=True)

In [50]:
def save_dataset_sample(
    pdb_id,
    chain_id,
    save_dir
):

    clear_mem()

    af_model = mk_afdesign_model(
        protocol="binder",
        debug=True
    )

    af_model.prep_inputs(
        pdb_filename=f"dataset/pdb/{pdb_id}.pdb",
        chain=chain_id,
        binder_len=20,
        rm_target_seq=False
    )

    af_model.set_seq("ACDEFGHIKLMNPQRSTVWY")

    af_model.predict(verbose=False)

    outputs = af_model.aux["debug"]["outputs"]

    pair_A = outputs["representations"]["pair"][:-20,-20:]
    pair_B = outputs["representations"]["pair"][-20:,:-20].swapaxes(0,1)

    pair_A = pair_A.reshape(pair_A.shape[0], -1)
    pair_B = pair_B.reshape(pair_B.shape[0], -1)

    x = np.concatenate([pair_A, pair_B], axis=-1)

    y = build_real_labels(
        f"dataset/pdb/{pdb_id}.pdb",
        chain_id=chain_id
    )

    n = min(len(x), len(y))

    x = x[:n]
    y = y[:n]

    save_data = {
        "x": x.astype(np.float32),
        "y": y.astype(np.float32)
    }

    save_path = f"{save_dir}/{pdb_id}_{chain_id}.pkl"

    with open(save_path, "wb") as f:

        pickle.dump(save_data, f)

    print("saved:", save_path)

In [51]:
for pdb_id, chain_id in train_targets:

    save_dataset_sample(
        pdb_id,
        chain_id,
        "dataset/train"
    )

saved: dataset/train/6w70_A.pkl
saved: dataset/train/1a3n_A.pkl
saved: dataset/train/1b38_A.pkl
saved: dataset/train/1h00_A.pkl


In [52]:
for pdb_id, chain_id in val_targets:

    save_dataset_sample(
        pdb_id,
        chain_id,
        "dataset/val"
    )

saved: dataset/val/2p16_A.pkl


In [53]:
train_dataset = AF2BindDataset(
    "dataset/train"
)

In [54]:
val_dataset = AF2BindDataset(
    "dataset/val"
)

In [55]:
from torch.utils.data import DataLoader

In [56]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False
)

In [57]:
model = AF2BindTransformer()

In [58]:
import torch.nn as nn
import torch.optim as optim

In [59]:
criterion = nn.BCELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [60]:
def train_epoch():

    model.train()

    total_loss = 0

    for x, y in train_loader:

        optimizer.zero_grad()

        pred = model(x)

        loss = criterion(pred, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [61]:
def validate():

    model.eval()

    total_loss = 0

    with torch.no_grad():

        for x, y in val_loader:

            pred = model(x)

            loss = criterion(pred, y)

            total_loss += loss.item()

    return total_loss / len(val_loader)

In [62]:
epochs = 20

for epoch in range(epochs):

    train_loss = train_epoch()

    val_loss = validate()

    print(
        f"Epoch {epoch+1}"
    )

    print(
        f"train loss = {train_loss:.4f}"
    )

    print(
        f"val loss = {val_loss:.4f}"
    )

    print("----------")

Epoch 1
train loss = 0.5529
val loss = 0.5674
----------
Epoch 2
train loss = 0.4435
val loss = 0.5604
----------
Epoch 3
train loss = 0.4354
val loss = 0.5752
----------
Epoch 4
train loss = 0.4368
val loss = 0.5693
----------
Epoch 5
train loss = 0.4333
val loss = 0.5498
----------
Epoch 6
train loss = 0.4270
val loss = 0.5465
----------
Epoch 7
train loss = 0.4112
val loss = 0.5460
----------
Epoch 8
train loss = 0.4087
val loss = 0.5424
----------
Epoch 9
train loss = 0.4014
val loss = 0.5385
----------
Epoch 10
train loss = 0.3931
val loss = 0.5369
----------
Epoch 11
train loss = 0.3884
val loss = 0.5405
----------
Epoch 12
train loss = 0.3791
val loss = 0.5496
----------
Epoch 13
train loss = 0.3684
val loss = 0.5438
----------
Epoch 14
train loss = 0.3589
val loss = 0.5613
----------
Epoch 15
train loss = 0.3433
val loss = 0.5520
----------
Epoch 16
train loss = 0.3332
val loss = 0.5650
----------
Epoch 17
train loss = 0.3296
val loss = 0.6221
----------
Epoch 18
train loss = 0

In [63]:
torch.save(
    model.state_dict(),
    "af2bind_transformer_real.pt"
)

In [64]:
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

In [65]:
def evaluate():

    model.eval()

    all_preds = []

    all_labels = []

    with torch.no_grad():

        for x, y in val_loader:

            pred = model(x)

            pred = pred.squeeze(0)

            all_preds.extend(
                pred.cpu().numpy()
            )

            all_labels.extend(
                y.squeeze(0).cpu().numpy()
            )

    all_preds = np.array(all_preds)

    all_labels = np.array(all_labels)

    ##################################################
    # binary prediction
    ##################################################

    binary_preds = (all_preds > 0.5).astype(int)

    ##################################################
    # metrics
    ##################################################

    auc = roc_auc_score(
        all_labels,
        all_preds
    )

    precision = precision_score(
        all_labels,
        binary_preds
    )

    recall = recall_score(
        all_labels,
        binary_preds
    )

    f1 = f1_score(
        all_labels,
        binary_preds
    )

    ##################################################

    print("ROC-AUC =", auc)

    print("Precision =", precision)

    print("Recall =", recall)

    print("F1 =", f1)

In [66]:
evaluate()

ROC-AUC = 0.5690267335004178
Precision = 0.7566371681415929
Recall = 1.0
F1 = 0.8614609571788413


In [ ]:
#@title **Run AF2BIND** 🔬
target_pdb = "6w70" #@param {type:"string"}
target_chain = "A" #@param {type:"string"}
#@markdown - Please indicate target pdb (or uniprot ID to download from AlphaFoldDB) and chain.
#@markdown - Leave pdb blank for custom upload prompt.
mask_sidechains = True
mask_sequence = False

target_pdb = target_pdb.replace(" ","")
target_chain = target_chain.replace(" ","")
if target_chain == "":
  target_chain = "A"

pdb_filename = get_pdb(target_pdb)

clear_mem()
af_model = mk_afdesign_model(protocol="binder", debug=True)
af_model.prep_inputs(pdb_filename=pdb_filename,
                     chain=target_chain,
                     binder_len=20,
                     rm_target_sc=mask_sidechains,
                     rm_target_seq=mask_sequence)

# split
r_idx = af_model._inputs["residue_index"][-20] + (1 + np.arange(20)) * 50
af_model._inputs["residue_index"][-20:] = r_idx.flatten()

af_model.set_seq("ACDEFGHIKLMNPQRSTVWY")
af_model.predict(verbose=False)

o = af2bind(af_model.aux["debug"]["outputs"],
            mask_sidechains=mask_sidechains)
pred_bind = o["p_bind"].copy()
pred_bind_aa = o["p_bind_aa"].copy()

#######################################################
labels = ["chain","resi","resn","p(bind)"]
data = []
for i in range(af_model._target_len):
  c = af_model._pdb["idx"]["chain"][i]
  r = af_model._pdb["idx"]["residue"][i]
  a = aa_order.get(af_model._pdb["batch"]["aatype"][i],"X")
  p = pred_bind[i]
  data.append([c,r,a,p])

df = pd.DataFrame(data, columns=labels)
df.to_csv('results.csv')

data_table.enable_dataframe_formatter()
df_sorted = df.sort_values("p(bind)",ascending=False, ignore_index=True).rename_axis('rank').reset_index()
display(data_table.DataTable(df_sorted, min_width=100, num_rows_per_page=15, include_index=False))

top_n = 15
top_n_idx = pred_bind.argsort()[::-1][:15]
pymol_cmd="select ch"+str(target_chain)+","
for n,i in enumerate(top_n_idx):
  p = pred_bind[i]
  c = af_model._pdb["idx"]["chain"][i]
  r = af_model._pdb["idx"]["residue"][i]
  pymol_cmd += f" resi {r}"
  if n < top_n-1:
    pymol_cmd += " +"

print("\n🧪Pymol Selection Cmd:")
print(pymol_cmd)

In [ ]:
#@title **Display Structure** (Colored by Confidence)
rescale_by_max_pbind = False
use_native_coordinates = True
show_ligand = False

if rescale_by_max_pbind:
  preds_adj = pred_bind.copy() / pred_bind.max()
else:
  preds_adj = pred_bind.copy()

# replace plddt and coordinates of prediction
L = af_model._target_len
aux = copy.deepcopy(af_model.aux["all"])
aux["plddt"][:,:L] = preds_adj
if show_ligand:
  af_model.save_pdb("output.pdb",aux={"all":aux})
else:
  aux["atom_mask"][:,L:] = 0
  x = {k:[] for k in ["aatype",
                      "residue_index",
                      "atom_positions",
                      "atom_mask",
                      "b_factors"]}
  asym_id = []
  for i in range(af_model._target_len):
    for k in ["aatype","atom_mask"]: x[k].append(aux[k][0,i])
    if use_native_coordinates:
      x["atom_positions"].append(af_model._pdb["batch"]["all_atom_positions"][i])
    else:
      x["atom_positions"].append(aux["atom_positions"][0,i])
    x["residue_index"].append(af_model._pdb["idx"]["residue"][i])
    x["b_factors"].append(x["atom_mask"][-1] * aux["plddt"][0,i] * 100.0)
    asym_id.append(af_model._pdb["idx"]["chain"][i])
  x = {k:np.array(v) for k,v in x.items()}

  # fix the chains
  (n,resnum_) = (0,None)
  pdb_lines = []
  for line in protein.to_pdb(protein.Protein(**x)).splitlines():
    if line[:4] == "ATOM":
      resnum = int(line[22:22+5])
      if resnum_ is None: resnum_ = resnum
      if resnum != resnum_:
        n += 1
        resnum_ = resnum
      pdb_lines.append("%s%s%4i%s" % (line[:21],asym_id[n],resnum,line[26:]))
  with open("output.pdb","w") as handle:
    handle.write("\n".join(pdb_lines))

hbondCutoff = 4.0
view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js',width=800,height=600)
pdb_str = open("output.pdb",'r').read()
view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
color_scheme = {'prop':'b','gradient': 'rwb','min':0,'max':100}
view.setStyle({'cartoon': {'colorscheme': color_scheme}})

# add sidechains
for i in range(af_model._target_len):
  c = af_model._pdb["idx"]["chain"][i]
  r = int(af_model._pdb["idx"]["residue"][i])
  p = pred_bind[i]
  if p > 0.5:
    view.addStyle({'and':[{'chain':c},{'resi':r},{'resn':["GLY","PRO"],'invert':True},{'atom':['C','O','N'],'invert':True}]},
                  {'stick':{'colorscheme':color_scheme,'radius':0.3}})
    view.addStyle({'and':[{'chain':c},{'resi':r},{'resn':"GLY"},{'atom':'CA'}]},
                  {'sphere':{'colorscheme':color_scheme,'radius':0.3}})
    view.addStyle({'and':[{'chain':c},{'resi':r},{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                  {'stick':{'colorscheme':color_scheme,'radius':0.3}})

view.setHoverable({}, True,
               '''function(atom,viewer,event,container){if(!atom.label){atom.label=viewer.addLabel(atom.chain+"/"+atom.resi+"/"+atom.resn+" "+(atom.b/100.0).toFixed(3),{position:atom,backgroundColor:'white',backgroundOpacity:0.75,borderColor:'black',borderThickness:2.0,fontColor:'black'});}}''',
               '''function(atom,viewer){if(atom.label){viewer.removeLabel(atom.label);delete atom.label;}}''')

view.zoomTo()
view.show()

def plot_plddt_legend(dpi=100):
  thresh = ['p(bind):','0.00','0.25','0.50','0.75','1.00']
  plt.figure(figsize=(1,0.1),dpi=dpi)
  ########################################
  for c in ["white","#FF0000","#FF8080","#FFFFFF","#8080FF","#0000FF"]:
    plt.bar(0, 0, color=c)
  plt.legend(thresh, frameon=False,
             loc='center', ncol=6,
             handletextpad=1,
             columnspacing=1,
             markerscale=0.5,)
  plt.axis(False)
  return plt
plot_plddt_legend().show()

In [ ]:
#@title **Download Predictions**
from google.colab import files
os.system(f"zip -r output.zip output.pdb results.csv")
files.download(f'output.zip')

In [ ]:
#@title **Activation analysis** (optional)
pbind_cutoff = 0.5 # @param ["0.0", "0.5", "0.9"] {type:"raw"}
blosum_map = list("CSTAGPDEQNHRKMILVWYF")
cs_label_list = list("ACDEFGHIKLMNPQRSTVWY")

indices_A_Y_mapping = np.array([cs_label_list.index(letter) for letter in blosum_map])
pred_bind_aa_blosum = pred_bind_aa[:,indices_A_Y_mapping]
filt = pred_bind > pbind_cutoff
pred_bind_aa_blosum = pred_bind_aa_blosum[filt]
res_labels = np.array(af_model._pdb["idx"]["residue"])[filt]
chain_labels = np.array(af_model._pdb["idx"]["chain"])[filt]

fig = px.imshow(pred_bind_aa_blosum.T,
                labels=dict(x="positions", y="amino acids", color="pref"),
                y=blosum_map,
                x=[f"{y}_{x}" for x,y in zip(res_labels,chain_labels)],
                zmin=-1,
                zmax=1,
                template="simple_white",
                color_continuous_scale=["red", "white", "blue"],
              )
fig.show()